In [ ]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

print("We are up and running!")

AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [ ]:
load_dotenv()

# Use the same model as simplest_agent.ipynb
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

In [ ]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

In [ ]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

In [ ]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [ ]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

In [ ]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [ ]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

In [ ]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

In [ ]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

In [ ]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

In [ ]:
print(final_response["output"]["message"]["content"][0]["text"])